# Lab: Building Your First LangChain Application with OpenRouter

## Overview

In this hands-on lab, you will build a simple Large Language Model (LLM) application using **LangChain** and **OpenRouter**.

The original version of this lab used the legacy `LLMChain` API. In this revised version, you will use **LangChain Expression Language (LCEL)**, which connects components with the `|` operator.

The application will follow this flow:

**User input → PromptTemplate → ChatOpenAI → StrOutputParser → Response**

### Learning Outcomes

By the end of this lab, you will be able to:

1. Install and import the packages needed for LangChain and OpenRouter.
2. Configure an OpenRouter API key securely.
3. Connect LangChain to an OpenRouter-hosted model.
4. Create a reusable `PromptTemplate`.
5. Build an LCEL chain using the `|` operator.
6. Execute a chain with `.invoke()`.
7. Turn the chain into a reusable Python function.

### Prerequisites

- Basic Python knowledge
- Access to Google Colab or Jupyter Notebook
- An OpenRouter API key
- No local LLM installation is required


## Step 1: Install the Required Packages

Run the following cell once in your notebook environment. The packages below provide the current LangChain core functionality and the OpenAI-compatible chat-model integration used by OpenRouter.

**Important:** This lab does **not** require `langchain-classic`, and it does not use the legacy `LLMChain` class.

In [1]:
%pip install -qU langchain langchain-openai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.7/163.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 20.9 MB/s eta 0:00:00


## Step 2: Import Libraries and Configure OpenRouter

OpenRouter provides an OpenAI-compatible API endpoint. LangChain's `ChatOpenAI` integration can therefore be configured with OpenRouter's base URL.

The API key is requested with `getpass()`, so it is not displayed on the screen.

In [2]:
from getpass import getpass

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Enter your OpenRouter API key when prompted.
OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key (hidden): ")

# OpenRouter's OpenAI-compatible API endpoint.
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

# Create the chat model.
# You can replace the model name with another model available in OpenRouter.
model = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0,
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_BASE_URL,
)

print("OpenRouter model configured successfully.")


Enter your OpenRouter API key (hidden): ··········
OpenRouter model configured successfully.


## Step 3: Create a PromptTemplate

A `PromptTemplate` separates the **instruction** from the **input value**.

In the example below, `{topic}` is a variable. When the chain runs, LangChain replaces `{topic}` with the value supplied by the user.

In [3]:
template = (
    "You are a concise technology instructor. "
    "Explain {topic} to a beginner in 4 sentences or fewer. "
    "Use a simple example when helpful."
)

prompt = PromptTemplate.from_template(template)

# Preview the prompt after inserting a value.
print(prompt.format(topic="vector databases"))


You are a concise technology instructor. Explain vector databases to a beginner in 4 sentences or fewer. Use a simple example when helpful.


## Step 4: Build the Chain Using LCEL

The legacy lab used `LLMChain`. The modern LangChain approach is **LCEL (LangChain Expression Language)**.

Use the `|` operator to connect runnable components:

```text
PromptTemplate → ChatOpenAI → StrOutputParser
```

`StrOutputParser` converts the model response into a normal Python string.

In [4]:
# Build the LCEL chain.
output_parser = StrOutputParser()

chain = prompt | model | output_parser

print("LCEL chain created successfully.")


LCEL chain created successfully.


## Step 5: Run the Chain

Use `.invoke()` to execute the chain. The input must contain a value for the `topic` variable defined in the prompt template.

In [5]:
result = chain.invoke({"topic": "LangChain"})

print(result)


LangChain is a framework designed to help developers build applications that utilize language models, like GPT, for various tasks. It allows you to easily connect language models with other data sources and tools, enabling more complex interactions. For example, you could create a chatbot that not only answers questions but also retrieves information from a database to provide more accurate responses. Essentially, LangChain streamlines the process of integrating language models into your applications.


## Step 6: Make the Chain Reusable

A useful application should not require us to repeat the same `invoke()` code. We can wrap the chain in a Python function.

In [6]:
def explain_topic(topic: str) -> str:
    """Return a concise explanation of a technology topic."""
    return chain.invoke({"topic": topic})

print(explain_topic("vector embeddings"))


Vector embeddings are a way to represent words, phrases, or items as numerical vectors in a high-dimensional space, allowing computers to understand their meanings and relationships. For example, the words "king" and "queen" can be represented as points in this space, where their distance reflects their similarity. This technique is commonly used in natural language processing to improve tasks like search and recommendation systems. Essentially, it transforms complex data into a format that machines can easily analyze and compare.


## Step 7: Try Your Own Topics

Modify the topics below and observe how the same chain can be reused for different inputs.

In [7]:
topics = [
    "large language models",
    "retrieval augmented generation",
    "vector databases",
]

for topic in topics:
    print(f"\n--- {topic} ---")
    print(explain_topic(topic))



--- large language models ---
Large language models are advanced AI systems designed to understand and generate human-like text. They learn from vast amounts of text data, allowing them to predict what words come next in a sentence. For example, if you start a sentence with "The cat sat on the," the model might suggest "mat" or "floor" based on patterns it has learned. Essentially, they can assist with writing, answering questions, and even having conversations.

--- retrieval augmented generation ---
Retrieval Augmented Generation (RAG) combines two processes: retrieving relevant information from a database and generating text based on that information. For example, if you ask a question about a historical event, RAG first searches a database for relevant facts and then uses those facts to create a coherent answer. This approach enhances the accuracy and relevance of the generated content by grounding it in real data. Essentially, it helps AI provide more informed and contextually ap

## Step 8: Understand the LCEL Components

| Component | Purpose |
|---|---|
| `PromptTemplate` | Defines the instructions and input variables |
| `ChatOpenAI` | Sends the formatted prompt to the OpenRouter model |
| `StrOutputParser` | Converts the model response into a string |
| `|` | Connects LangChain runnable components |
| `.invoke()` | Executes the chain with an input |

The complete application can therefore be represented as:

```text
topic
  ↓
PromptTemplate
  ↓
ChatOpenAI / OpenRouter
  ↓
StrOutputParser
  ↓
Python string
```

## Step 9: Lab Exercise

Modify the application so that it acts as a **course tutor** rather than a general technology explainer.

### Requirements

1. Change the prompt so the model acts as a course tutor.
2. Add a second input variable called `{level}`.
3. Ask the model to explain the topic at the requested level.
4. Rebuild the LCEL chain.
5. Test the chain with at least three different topics.

For example, your prompt could accept:

- `topic = "cloud computing"`
- `level = "beginner"`

### Starter Code

Complete the following code yourself.

In [ ]:
# TODO: Create a prompt with {topic} and {level}
# TODO: Create the PromptTemplate
# TODO: Build the LCEL chain
# TODO: Invoke the chain

# Example structure:
# tutor_template = "... {topic} ... {level} ..."
# tutor_prompt = PromptTemplate.from_template(tutor_template)
# tutor_chain = tutor_prompt | model | output_parser
# print(tutor_chain.invoke({"topic": "cloud computing", "level": "beginner"}))


## Step 10: Troubleshooting

### `ModuleNotFoundError: No module named 'langchain.chains'`

This occurs when code tries to import the legacy `LLMChain` API from `langchain.chains`. This lab intentionally does not use that import. Use:

```python
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
```

and construct the chain with:

```python
chain = prompt | model | output_parser
```

### Authentication errors

Check that your OpenRouter API key is valid and that it has access to the selected model.

### Model errors

If the selected model is unavailable, replace the `model=` value with a model currently available in your OpenRouter account.

### Do not expose your API key

Never put an API key directly into code that you will submit to GitHub or share with others. The `getpass()` approach used in this lab keeps the key out of the visible notebook code.

## Lab Summary

In this lab, you built a complete LangChain application using OpenRouter without relying on the legacy `LLMChain` API.

The key pattern to remember is:

```python
chain = prompt | model | output_parser
result = chain.invoke({"topic": "LangChain"})
```

This LCEL pattern provides a simple foundation for more advanced LangChain applications involving multiple steps, tools, retrieval, agents, and structured outputs.

### Key Takeaways

- Use `PromptTemplate` to separate prompts from input values.
- Use `ChatOpenAI` to connect LangChain to an OpenAI-compatible provider such as OpenRouter.
- Use LCEL and the `|` operator to compose chains.
- Use `.invoke()` to execute a chain.
- Use `StrOutputParser` when a plain text result is required.
- Keep API keys out of source code and shared repositories.